# Signed Jumps 03 — do "bubbles" (aggressor activity) predict when a jump happens?

Protocol: `discussions/Signed_Jumps/005_claude_sj03_preregistration.md` (fixed before this run).

Same jumps as SJ 01. Models (b) → (bf) and (e) → (f) add order-flow heat readings (trade/volume acceleration, |taker imbalance|).
Gates: **F1** (f) > (e); **F2** 15-min forecast (f) > (e); **F3** (f) > (e + flow-implied volatility) — the only gate that shows boil-specific heat.

In [ ]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import signed_jumps as sj, signed_jumps_flow as sf
for m in (sj,sf):importlib.reload(m)
assert sj.VERSION.startswith('SJ01-0.1') and sf.VERSION=='SJ03-0.1.0'
SYNTHETIC=os.getenv('SJ03_SYNTHETIC','0')=='1'
p=sj.Params();fp=sf.FlowParams()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='sj03-');OUT=Path(tmp.name)/'03'
    p=sj.Params(n_boot=200,min_test_jumps_per_sign=20,mc_paths=100);fp=sf.FlowParams(n_boot=200)
else:
    OUT=ROOT/'notebooks'/'Signed_Jumps'/'03'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
if not SYNTHETIC:
    chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_signed_jumps_flow.py'),str(ROOT/'tests'/'test_signed_jumps.py')],
                       cwd=str(ROOT),env=env,capture_output=True,text=True)
    print(chk.stdout[-600:]);print(chk.stderr[-400:]);assert chk.returncode==0,'tests failed'
print(json.dumps(sf.registration(fp),indent=1))

## Data: SJ 01 engine (unchanged) + order flow

In [ ]:
T0=time.time()
if SYNTHETIC:
    S0,F0=sj.simulate(n_days=60,n_assets=len(p.assets),eta=((0.2,0.05),(0.05,0.3)),half_life=1,seed=12,df=8,base_rate=8e-4,
                      flow=True,heat_link=1.0,flow_heat=1.0,flow_vol=0.5)
    S={a:s for a,s in zip(p.assets,S0.values())};FR={a:f for a,f in zip(p.assets,F0.values())}
else:
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
    S={A:sj.load_spot_1m(BIN/'basis'/'spot'/f'{A}USDT_spot_1m.parquet') for A in p.assets}
assets,pan,dev_end=sj.prepare(S,p)
if SYNTHETIC:
    flows={a.name:FR[a.name].reindex(a.time) for a in assets}
else:
    flows={a.name:sf.load_spot_flow(BIN/'basis'/'spot'/f'{a.name}USDT_spot_1m.parquet',a.time) for a in assets}
    q01=pd.read_csv(ROOT/'notebooks'/'Signed_Jumps'/'01'/'01_data_quality.csv').set_index('asset')
    for i,a in enumerate(assets):
        m=pan.asset_idx==i
        for s in sj.SIGNS:
            for per,mm in (('dev',pan.dev),('test',~pan.dev)):
                assert int(pan.J[s][m&mm].sum())==int(q01.loc[a.name,f'{s}_{per}']),(a.name,s,per)
    print('jump counts reproduce SJ 01')
pf=sf.attach_flow(assets,pan,flows,fp)
cov=pd.DataFrame({'asset':[a.name for a in assets],'flow_missing_share':[float(flows[a.name].isna().any(axis=1).mean()) for a in assets],
                  'panel_rows_sj01':[int((pan.asset_idx==i).sum()) for i in range(len(assets))],
                  'panel_rows_flow':[int((pf.asset_idx==i).sum()) for i in range(len(assets))]})
display(cov);cov.to_csv(OUT/'03_coverage.csv',index=False)
print('flow covariates:',pf.extra_names,'| DEV scaler',pf.extra_scaler,f'| {time.time()-T0:.0f}s')

## Fit (b), (bf), (e), (f) on DEV; F1 on TEST

In [ ]:
T1=time.time()
fits,E=sf.fit_models(assets,pf,p,fp)
print(f'fits {time.time()-T1:.0f}s; converged:',{k:f.converged for k,f in fits.items()})
cmp,nj=sf.compare(fits,pf,E,p)
print('TEST jumps',nj);display(cmp[['test','delta_per_row','q025','q975','n_days','status']])
dev=pf.dev;se_rows=[]
for m in ('bf','f'):
    X=sj.design(pf,m)[dev].astype(float)
    for s in sj.SIGNS:
        f=fits[(m,s)];Ex=sj.excitation_design(E[fp.sj01_half_life],m,s);Ex=None if Ex is None else Ex[dev]
        se=sj.cluster_se(f,X,Ex,pf.J[s][dev],pf.day[dev])
        names=sj.design_names(pf,m,p)+(['eta_from_up','eta_from_down'] if Ex is not None else [])
        for n,v,e in zip(names,np.concatenate([f.b,f.eta]),se):se_rows.append({'model':m,'child_sign':s,'param':n,'estimate':v,'cluster_se':e})
se_tab=pd.DataFrame(se_rows);se_tab['t']=se_tab.estimate/se_tab.cluster_se;se_tab.to_csv(OUT/'03_param_se.csv',index=False)
display(se_tab[se_tab.param.isin(list(pf.extra_names)+['TV10','TV60'])].round(4))

## F3: is it boil-specific, or just a sharper volatility thermometer?

In [ ]:
vp,vinfo=sf.vhat_panel(assets,pf,p)
print(json.dumps(vinfo,indent=1))
f3,fits_v=sf.compare_f3(fits,E,vp,assets,p,pf)
cmp=pd.concat([cmp,pd.DataFrame([f3])],ignore_index=True);cmp.to_csv(OUT/'03_tests.csv',index=False)
display(cmp[['test','delta_per_row','q025','q975','status']])

## F2: 15-minute jump forecast; lift table

In [ ]:
T2=time.time()
hz,pr,ro=sf.horizon_compare(fits,assets,pf,p)
print(f'{time.time()-T2:.0f}s');display(hz[['test','horizon_min','delta_per_row','q025','q975','status']]);hz.to_csv(OUT/'03_horizon.csv',index=False)
lift=sf.lift_table(pr,ro,pf,p.primary_horizon);lift.to_csv(OUT/'03_lift_15min.csv',index=False)
display(lift.pivot_table(index='dec',columns='model',values=['realised','lift'],aggfunc='first').round(4))

## Event-time profile: is there a build-up of activity before jumps? (descriptive)

In [ ]:
prof=sf.event_profile(assets,pf,flows,fp);prof.to_csv(OUT/'03_event_profile.csv',index=False)
display(prof.pivot_table(index='k',columns=['series'],values='excess',aggfunc='mean').loc[[-30,-20,-10,-5,-3,-2,-1,0,1,2,5,10]].round(3))

## Decision

In [ ]:
dec=sf.decide(cmp,hz,f3,p)
dec.update(flow_vhat_info=vinfo,test_jumps=nj,dev_end=str(dev_end),synthetic=SYNTHETIC,
           top1pct_15min={m:lift[(lift.model==m)&(lift.dec.astype(str)=='top1%')][['realised','lift']].iloc[0].to_dict() for m in ('b','e','f')})
print(json.dumps(dec,indent=1,default=float))
if not SYNTHETIC:
    (OUT/'03_decision.json').write_text(json.dumps(dec,indent=1,default=float))
    (OUT/'03_manifest.json').write_text(json.dumps({'registration':sf.registration(fp),'runtime_s':time.time()-T0,'rows':int(len(pf.r))},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')